In [1]:
import pandas as pd

data = pd.read_csv('dataset.csv')

data.head()

,id,project_a,project_b,weight_a,weight_b
0,2,https://github.com/prettier-solidity/prettier-...,https://github.com/nomicfoundation/hardhat,0.101669,0.898331
1,3,https://github.com/prettier-solidity/prettier-...,https://github.com/consensys/teku,0.669446,0.330554
2,4,https://github.com/prettier-solidity/prettier-...,https://github.com/ethereum/solidity,0.449022,0.550978
3,5,https://github.com/prettier-solidity/prettier-...,https://github.com/ethereum/remix-project,0.498396,0.501604
4,6,https://github.com/prettier-solidity/prettier-...,https://github.com/ethereum/go-ethereum,0.272503,0.727497


In [2]:
# find which project appears in most rows  
value_counts_a = data.project_a.value_counts()
value_counts_b = data.project_b.value_counts()

value_counts_a.rename("project")
value_counts_b.rename("project")

# join two series, add the counts if there are duplicates
value_counts = value_counts_a.add(value_counts_b, fill_value=0)

value_counts.sort_values(inplace=True, ascending=False)

value_counts

https://github.com/sindresorhus/type-fest    66.0
https://github.com/motdotla/dotenv           66.0
https://github.com/zloirock/core-js          65.0
https://github.com/go-task/slim-sprig        64.0
https://github.com/postcss/postcss           64.0
                                             ... 
https://github.com/quic-go/quic-go            9.0
https://github.com/erigontech/erigon          8.0
https://github.com/ethereum/solc-js           8.0
https://github.com/grandinetech/grandine      8.0
https://github.com/ethereum/web3.py           1.0
Length: 117, dtype: float64

In [3]:
df_list = []

In [4]:
for i in range(len(value_counts)):
    start_project = value_counts.index[i]

    # find all projects that are connected to start_project
    fd = data[(data['project_a'] == start_project) | (data['project_b'] == start_project)]
    start_score = 1


    current_df = pd.DataFrame([{
        'project': start_project,
        'score': start_score
    }])

    for index, row in fd.iterrows():
        if row['project_a'] == start_project:
            current_df.loc[len(current_df)] = {
                'project': row['project_b'],
                'score': row['weight_b'] * start_score / row['weight_a']
            }
        else:
            current_df.loc[len(current_df)] = {
                'project': row['project_a'],
                'score': row['weight_a'] * start_score / row['weight_b']
            }

    processed = [start_project]

    while len(current_df) < len(value_counts):
        current_counts = value_counts[value_counts.index.isin(current_df.project) & ~value_counts.index.isin(processed)]

        current_counts.sort_values(ascending=False, inplace=True)

        start_project = current_counts.index[0]

        # find all projects that are connected to start_project
        fd = data[(data['project_a'] == start_project) | (data['project_b'] == start_project)]

        for index, row in fd.iterrows():
            if row['project_a'] == start_project:
                if row['project_b'] not in current_df.project.values:
                    current_df.loc[len(current_df)] = {
                        'project': row['project_b'],
                        'score': row['weight_b'] * current_df[current_df.project == start_project].score.values[0] / row['weight_a']
                    }
            else:
                if row['project_a'] not in current_df.project.values:
                    current_df.loc[len(current_df)] = {
                        'project': row['project_a'],
                        'score': row['weight_a'] * current_df[current_df.project == start_project].score.values[0] / row['weight_b']
                    }

        processed.append(start_project)
    
    # L2 normalize
    current_df['score'] = current_df['score'] / current_df['score'].pow(2).sum()**0.5

    df_list.append(current_df)

len(df_list)


117

In [5]:
df = pd.concat(df_list).groupby('project').score.mean().sort_values(ascending=False)

df.head()

project
https://github.com/bradfitz/iter                   0.730103
https://github.com/coinbase/coinbase-wallet-sdk    0.254336
https://github.com/yahoo/serialize-javascript      0.239836
https://github.com/fb55/entities                   0.123352
https://github.com/webpack/webpack                 0.107657
Name: score, dtype: float64

In [6]:
# reproduce the dataset with the scores

for index, row in data.iterrows():
    score_a = df[row['project_a']]
    score_b = df[row['project_b']]

    pred_a = score_a / (score_a + score_b)
    pred_b = score_b / (score_a + score_b)

    data.loc[index, 'pred_a'] = pred_a
    data.loc[index, 'pred_b'] = pred_b

    data.loc[index, 'squared_error'] = (pred_a - row['weight_a']) ** 2

print(data.squared_error.mean())
data.head()

0.0173419588805486


,id,project_a,project_b,weight_a,weight_b,pred_a,pred_b,squared_error
0,2,https://github.com/prettier-solidity/prettier-...,https://github.com/nomicfoundation/hardhat,0.101669,0.898331,0.073079,0.926921,0.000817
1,3,https://github.com/prettier-solidity/prettier-...,https://github.com/consensys/teku,0.669446,0.330554,0.633591,0.366409,0.001286
2,4,https://github.com/prettier-solidity/prettier-...,https://github.com/ethereum/solidity,0.449022,0.550978,0.072381,0.927619,0.141859
3,5,https://github.com/prettier-solidity/prettier-...,https://github.com/ethereum/remix-project,0.498396,0.501604,0.412383,0.587617,0.007398
4,6,https://github.com/prettier-solidity/prettier-...,https://github.com/ethereum/go-ethereum,0.272503,0.727497,0.063588,0.936412,0.043645


In [7]:
df = df.to_frame().reset_index()

In [8]:
df.head()

,project,score
0,https://github.com/bradfitz/iter,0.730103
1,https://github.com/coinbase/coinbase-wallet-sdk,0.254336
2,https://github.com/yahoo/serialize-javascript,0.239836
3,https://github.com/fb55/entities,0.123352
4,https://github.com/webpack/webpack,0.107657


In [9]:
test = pd.read_csv('test.csv')

test.head()

,id,project_a,project_b
0,1,https://github.com/prettier-solidity/prettier-...,https://github.com/ethers-io/ethers.js
1,8,https://github.com/prettier-solidity/prettier-...,https://github.com/bluealloy/revm
2,13,https://github.com/prettier-solidity/prettier-...,https://github.com/paradigmxyz/reth
3,15,https://github.com/prettier-solidity/prettier-...,https://github.com/ipfs/js-ipfs
4,18,https://github.com/prysmaticlabs/prysm,https://github.com/consensys/teku


In [10]:
# calculate the relative weight

for index, row in test.iterrows():
    score_a = df[df['project'] == row['project_a']]['score'].values[0]
    score_b = df[df['project'] == row['project_b']]['score'].values[0]

    pred_a = score_a / (score_a + score_b)
    pred_b = score_b / (score_a + score_b)

    test.loc[index, 'pred'] = pred_a
test.head()

,id,project_a,project_b,pred
0,1,https://github.com/prettier-solidity/prettier-...,https://github.com/ethers-io/ethers.js,0.030833
1,8,https://github.com/prettier-solidity/prettier-...,https://github.com/bluealloy/revm,0.010502
2,13,https://github.com/prettier-solidity/prettier-...,https://github.com/paradigmxyz/reth,0.062172
3,15,https://github.com/prettier-solidity/prettier-...,https://github.com/ipfs/js-ipfs,0.095588
4,18,https://github.com/prysmaticlabs/prysm,https://github.com/consensys/teku,0.998560


In [11]:
submission = test[['id', 'pred']]

# 11 decimal places only
submission.to_csv('submissions/baseline_new.csv', index=False, float_format='%.11f')